In [ ]:
import numpy as np
from typing import Optional, List

from initializers import initialize_weights
from activations import get_activation, get_activation_derivative
from losses import get_loss_function, get_loss_derivative, l2_regularization
from optimizers import get_optimizer

In [21]:
class DenseLayer:
    """Holds W, b, activation, and caches. Does NOT perform forward/backward."""
    def __init__(self, input_size, output_size, activation='relu',
                 weight_init='xavier', seed=None):

        self.W, self.b = initialize_weights(
            input_size,
            output_size,
            method=weight_init,
            seed=seed
        )

        self.activation = activation
        self.activation_cache = {
            'A_prev': None,
            'Z': None,
            'A': None
        }

        # Gradients
        self.dW = None
        self.db = None


In [26]:

class NeuralNetwork:
    # Fully-connected feedforward neural network implemented with NumPy
    # Supports: configurable layers, multiple activations, optimizers, L2 regularization
    
    def __init__(
        self,
        input_size: int,
        hidden_layers: List[int],
        output_size: int,
        activation: str = 'relu',
        output_activation: str = 'softmax',
        learning_rate: float = 0.01,
        optimizer: str = 'sgd',
        weight_init: str = 'xavier',
        l2_lambda: float = 0.0,
        random_seed: Optional[int] = None
    ):
        # Initialize network: create layers, set up optimizer, store hyperparameters
        self.layers = []
        input_dim = input_size

        # Create hidden layers
        for hidden_units in hidden_layers:
            layer = DenseLayer(
                input_size=input_dim,
                output_size=hidden_units,
                activation=activation,
                weight_init=weight_init,
                seed=random_seed
            )
            self.layers.append(layer)
            input_dim = hidden_units 

        # Output layer
        self.layers.append(DenseLayer(
            input_size=input_dim,
            output_size=output_size,
            activation=output_activation,
            weight_init=weight_init,
            seed=random_seed
        ))
        
        # Store hyperparameters
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        self.activation = activation
        self.output_activation = output_activation
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer
        self.l2_lambda = l2_lambda
        
        # Initialize optimizer
        self.optimizer = get_optimizer(optimizer, learning_rate=learning_rate)
        
        # Store loss function (default to cross_entropy for classification)
        self.loss_function = 'cross_entropy'
        
        # Store last predictions and loss for debugging
        self.last_predictions = None
        self.last_loss = None
    
    
    # ------------------------------------------------------------------
    # FORWARD
    # ------------------------------------------------------------------

    def forward(self, X: np.ndarray) -> np.ndarray:

        A = X

        for layer in self.layers:

            # Cache previous activation
            layer.activation_cache['A_prev'] = A

            # Linear
            Z = A @ layer.W + layer.b
            layer.activation_cache['Z'] = Z

            # Activation
            act = get_activation(layer.activation)
            A = act(Z)

            # Cache activation
            layer.activation_cache['A'] = A

        return A

    # ------------------------------------------------------------------
    # BACKWARD
    # ------------------------------------------------------------------

    def backward(self, X: np.ndarray, y: np.ndarray, y_pred=None):

        if y_pred is None:
            y_pred = self.forward(X)

        loss_deriv_fn = get_loss_derivative(self.loss_function)
        dA = loss_deriv_fn(y_pred, y)

        # Backprop layers in reverse
        for layer in reversed(self.layers):

            Z = layer.activation_cache['Z']
            A_prev = layer.activation_cache['A_prev']

            # dZ = dA * activation'(Z)
            activation_grad = get_activation_derivative(layer.activation)
            dZ = dA * activation_grad(Z)

            # Gradients
            layer.dW = A_prev.T @ dZ
            layer.db = np.sum(dZ, axis=0)

            # L2 regularization
            if self.l2_lambda > 0:
                layer.dW += self.l2_lambda * layer.W

            # Next gradient
            dA = dZ @ layer.W.T

    # ------------------------------------------------------------------
    # UPDATE
    # ------------------------------------------------------------------

    def update_weights(self) -> None:
        # Update all weights and biases using the optimizer
        
        # Collect parameters and gradients from all layers
        params = {}
        grads = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i+1}'] = layer.W
            params[f'b{i+1}'] = layer.b
            grads[f'W{i+1}'] = layer.dW
            grads[f'b{i+1}'] = layer.db
        
        # Apply optimizer update rule
        updated_params = self.optimizer.update(params, grads)
        
        # Update layer weights and biases with new values
        for i, layer in enumerate(self.layers):
            layer.W = updated_params[f'W{i+1}']
            layer.b = updated_params[f'b{i+1}']
    
    
    # ------------------------------------------------------------------
    # COMPUTE LOSS
    # ------------------------------------------------------------------

    def compute_loss(self, y_pred: np.ndarray, y_true: np.ndarray) -> float:
        # Compute total loss: data loss + L2 regularization
        # y_pred: predictions, y_true: true labels -> returns: scalar loss
        
        # Compute data loss (e.g., cross-entropy)
        loss_func = get_loss_function(self.loss_function)
        data_loss = loss_func(y_pred, y_true)
        
        # Add L2 regularization term if specified
        if self.l2_lambda > 0:
            weights = [layer.W for layer in self.layers]
            reg_loss = l2_regularization(weights, self.l2_lambda)
            total_loss = data_loss + reg_loss
        else:
            total_loss = data_loss
        
        return total_loss
    

    # ------------------------------------------------------------------
    # TRAINING STEP
    # ------------------------------------------------------------------

    def train_step(self, X_batch: np.ndarray, y_batch: np.ndarray) -> float:
        # One complete training step: forward -> backward -> update
        # Returns loss value for this batch
        
        # Forward pass: compute predictions
        y_pred = self.forward(X_batch)
        
        # Compute loss
        loss = self.compute_loss(y_pred, y_batch)
        
        # Backward pass: compute gradients (pass y_pred to avoid redundant forward)
        self.backward(X_batch, y_batch, y_pred=y_pred)
        
        # Update weights using computed gradients
        self.update_weights()
        
        # Store for debugging/monitoring
        self.last_predictions = y_pred
        self.last_loss = loss
        
        return loss

    # ------------------------------------------------------------------
    # PREDICT
    # ------------------------------------------------------------------

    def predict(self, X: np.ndarray) -> np.ndarray:
        # Predict class labels: get probabilities and return argmax
        # X: (batch_size, input_size) -> returns: (batch_size,) class indices
        probabilities = self.predict_proba(X)
        predictions = np.argmax(probabilities, axis=1)
        return predictions
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        # Get prediction probabilities (forward pass without training)
        # X: (batch_size, input_size) -> returns: (batch_size, output_size) probabilities
        probabilities = self.forward(X)
        return probabilities
    
    
    # ------------------------------------------------------------------
    # PARAMETERS
    # ------------------------------------------------------------------

    def get_params(self) -> dict:
        # Get all model parameters (weights and biases) as dictionary
        # Returns: {'W1': weights, 'b1': biases, 'W2': ..., ...}
        params = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i+1}'] = layer.W.copy()
            params[f'b{i+1}'] = layer.b.copy()
        return params
    
    def set_params(self, params: dict) -> None:
        # Set model parameters from dictionary
        # params: {'W1': weights, 'b1': biases, 'W2': ..., ...}
        for i, layer in enumerate(self.layers):
            layer.W = params[f'W{i+1}'].copy()
            layer.b = params[f'b{i+1}'].copy()


## train.py

In [43]:
# Main training script for neural network experiments with WandB logging

import numpy as np
import sys
import os

from data_loader import load_fashion_mnist, load_cifar10, preprocess_data, create_mini_batches, train_val_split
from utils import accuracy_score, plot_training_curves, set_random_seed
import wandb


def train_epoch(model, X_train, y_train, batch_size):
    # Train for one epoch: create mini-batches, train on each, compute average loss and accuracy
    batches = create_mini_batches(X_train, y_train, batch_size=batch_size, shuffle=True)
    
    epoch_losses = []
    epoch_predictions = []
    epoch_labels = []
    
    for X_batch, y_batch in batches:
        # Train on batch
        loss = model.train_step(X_batch, y_batch)
        epoch_losses.append(loss)
        
        # Get predictions for accuracy
        predictions = model.predict(X_batch)
        epoch_predictions.append(predictions)
        
        # Convert y_batch to class indices for accuracy calculation
        if y_batch.ndim > 1:
            y_batch_indices = np.argmax(y_batch, axis=1)
        else:
            y_batch_indices = y_batch
        epoch_labels.append(y_batch_indices)
    
    # Compute average loss and accuracy
    avg_loss = np.mean(epoch_losses)
    all_predictions = np.concatenate(epoch_predictions)
    all_labels = np.concatenate(epoch_labels)
    accuracy = accuracy_score(all_predictions, all_labels)
    
    return avg_loss, accuracy


def evaluate(model, X_val, y_val):
    # Evaluate model on validation set: make predictions, compute loss and accuracy
    # Get probability predictions and class predictions
    y_pred_proba = model.predict_proba(X_val)
    y_pred = model.predict(X_val)
    
    # Compute loss
    loss = model.compute_loss(y_pred_proba, y_val)
    
    # Compute accuracy
    # Convert y_val to class indices if one-hot encoded
    if y_val.ndim > 1 and y_val.shape[1] > 1:
        y_val_indices = np.argmax(y_val, axis=1)
    else:
        y_val_indices = y_val
    
    accuracy = accuracy_score(y_pred, y_val_indices)
    
    return loss, accuracy


def train(config):
    # Complete training pipeline: WandB init, load data, create model, train, log metrics
    
    # Initialize Weights & Biases for experiment tracking (optional)
    use_wandb = config.get('use_wandb', True)
    run = None
    if use_wandb:
        try:
            # Initialize WandB run
            run = wandb.init(
                project=config.get('project_name', 'neural-network-numpy'),
                name=config.get('experiment_name', 'baseline'),
                entity=config.get('entity', None),  # Optional: set if using team account
                config=config,
                resume='allow'  # Allow resuming if run already exists
            )
            print(f"✅ WandB initialized: {run.url}")
        except Exception as e:
            print(f"⚠️  Warning: WandB initialization failed ({e})")
            print("   Continuing without WandB logging. To fix: run 'wandb login'")
            use_wandb = False
            run = None
    
    # Set random seed
    set_random_seed(config['random_seed'])
    
    # Load dataset
    data_dir = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'data')
    if config['dataset'] == 'fashion_mnist':
        X_train_full, y_train_full, X_test, y_test = load_fashion_mnist(data_dir)
        input_size = 784  # 28*28
    elif config['dataset'] == 'cifar10':
        X_train_full, y_train_full, X_test, y_test = load_cifar10(data_dir)
        input_size = 3072  # 32*32*3
    else:
        raise ValueError(f"Unknown dataset: {config['dataset']}")
    
    # Preprocess data
    X_train_full, y_train_full = preprocess_data(
        X_train_full, y_train_full,
        num_classes=config['output_size'],
        flatten=True,
        normalize=True
    )
    X_test, y_test = preprocess_data(
        X_test, y_test,
        num_classes=config['output_size'],
        flatten=True,
        normalize=True
    )
    
    # Split into train/val
    X_train, X_val, y_train, y_val = train_val_split(
        X_train_full, y_train_full,
        val_split=config['val_split'],
        random_seed=config['random_seed']
    )
    
    print(f"Dataset: {config['dataset']}")
    print(f"Train samples: {X_train.shape[0]}, Val samples: {X_val.shape[0]}, Test samples: {X_test.shape[0]}")
    
    # Create model
    model = NeuralNetwork(
        input_size=input_size,
        hidden_layers=config['hidden_layers'],
        output_size=config['output_size'],
        activation=config['activation'],
        output_activation=config['output_activation'],
        learning_rate=config['learning_rate'],
        optimizer=config['optimizer'],
        weight_init=config['weight_init'],
        l2_lambda=config['l2_lambda'],
        random_seed=config['random_seed']
    )
    
    # Training loop
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    best_val_acc = 0.0
    best_model_params = None
    
    print("\nStarting training...")
    print(f"Epochs: {config['num_epochs']}, Batch size: {config['batch_size']}, Learning rate: {config['learning_rate']}")
    print("-" * 80)
    
    for epoch in range(config['num_epochs']):
        # Train for one epoch
        train_loss, train_acc = train_epoch(model, X_train, y_train, config['batch_size'])
        
        # Evaluate on validation set
        val_loss, val_acc = evaluate(model, X_val, y_val)
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_params = model.get_params()
        
        # Log to WandB
        if use_wandb and run is not None:
            run.log({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'val_acc': val_acc
            })
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{config['num_epochs']} | "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Load best model
    if best_model_params is not None:
        model.set_params(best_model_params)
        print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    
    # Evaluate on test set
    test_loss, test_acc = evaluate(model, X_test, y_test)
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
    
    # Log test metrics
    if use_wandb and run is not None:
        run.log({
            'test_loss': test_loss,
            'test_acc': test_acc
        })
    
    # Save model
    os.makedirs('results/models', exist_ok=True)
    model_path = f"results/models/{config['experiment_name']}_best.pkl"
    import pickle
    with open(model_path, 'wb') as f:
        pickle.dump(best_model_params, f)
    print(f"Model saved to {model_path}")
    
    # Finish WandB run
    if use_wandb and run is not None:
        run.finish()
        print(f"✅ WandB run completed. View at: {run.url}")


def main():
    # Main function to run training
    
    # Default configuration
    config = {
        # Dataset
        'dataset': 'fashion_mnist',  # or 'cifar10'
        
        # Model architecture
        'input_size': 784,  # 28*28 for Fashion-MNIST (will be overridden based on dataset)
        'hidden_layers': [128, 64],
        'output_size': 10,
        
        # Activation and loss
        'activation': 'relu',
        'output_activation': 'softmax',
        'loss': 'cross_entropy',
        
        # Training hyperparameters
        'num_epochs': 50,
        'batch_size': 32,
        'learning_rate': 0.001,
        
        # Optimization
        'optimizer': 'sgd',  # 'sgd', 'momentum', 'rmsprop', 'adam' (use 'sgd' for now)
        
        # Regularization
        'l2_lambda': 0.0001,
        
        # Initialization
        'weight_init': 'xavier',  # 'random', 'xavier', 'he'
        
        # Other
        'val_split': 0.2,
        'random_seed': 42,
        'project_name': 'neural-network-numpy',
        'experiment_name': 'baseline',
        'use_wandb': True,  # Set to False to disable WandB
        'entity': 'makssuppras1-danmarks-tekniske-universitet-dtu'  # Your WandB entity
    }
    
    # Parse command line arguments to override config if needed
    import argparse
    parser = argparse.ArgumentParser(description='Train neural network')
    parser.add_argument('--dataset', type=str, default=config['dataset'])
    parser.add_argument('--epochs', type=int, default=config['num_epochs'])
    parser.add_argument('--batch-size', type=int, default=config['batch_size'])
    parser.add_argument('--lr', type=float, default=config['learning_rate'])
    parser.add_argument('--optimizer', type=str, default=config['optimizer'])
    parser.add_argument('--name', type=str, default=config['experiment_name'])
    parser.add_argument('--no-wandb', action='store_true', help='Disable WandB logging')
    
    args = parser.parse_args()
    
    # Update config with command line arguments
    config['dataset'] = args.dataset
    config['num_epochs'] = args.epochs
    config['batch_size'] = args.batch_size
    config['learning_rate'] = args.lr
    config['optimizer'] = args.optimizer
    config['experiment_name'] = args.name
    if args.no_wandb:
        config['use_wandb'] = False
    
    # Run training
    train(config)
    
    print("Training completed!")


if __name__ == '__main__':
    main()



ModuleNotFoundError: No module named 'wandb'